# Setup

## Import modules

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import torch

# Import pipeline modules
from scoring import score_image
from selection import select
from metrics import MetricsTracker, image_metrics_dict
from generation import UnconditionalGenerator
from utils import save_image, save_image_batch, set_seed

# Config
OUTPUT_DIR = Path("outputs")
IMAGES_DIR = OUTPUT_DIR / "images"
METRICS_CSV = OUTPUT_DIR / "metrics.csv"

# Experiment parameters
NUM_PROMPTS = 5
NUM_TRIALS = 1
ROUNDS = 2        # r
BATCH_SIZE = 4    # B
SEED = 42

# Chosen strategies
SELECTION_STRATEGY = "argmax"

# Baseline scoring rubric (example)
# CLIP_RUBRIC = {"type": "clip", "text": "A photorealistic portrait of a dog", "weight": 1.0}
BRIGHTNESS_RUBRIC = {"type": "brightness", "weight": 1.0}

set_seed(SEED)


ModuleNotFoundError: No module named 'pandas'

## Load diffusion model

In [ ]:
# Cell 2: load diffusion pipeline (adjust model_name as needed)
from diffusers import StableDiffusionPipeline

MODEL_NAME = "runwayml/stable-diffusion-v2-1"  # TODO: replace with SD3 HF model if available
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16 if device=="cuda" else torch.float32
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # memory-friendly

## Load dataset

In [ ]:
# Mock dataset
dataset = [
    {"prompt_id": f"p{i}+1", 
    "prompt": f"A photograph of {i+1} dogs",
    "rubric": BRIGHTNESS_RUBRIC}
    for i in range(NUM_PROMPTS)
]

# Run simulation

In [ ]:
from random import sample


metrics = MetricsTracker()
policy = UnconditionalGenerator(pipe)

for trial in range(NUM_TRIALS):
    print("="*100)
    print(f"BEGINNING TRIAL {trial}")
    print("="*100)

    trial_id = f"trial{trial}"

    for item in tqdm(dataset, desc="prompts"):
        prompt_id = item["prompt_id"]
        prompt = item["prompt"]
        rubric = item["rubric"]

        print(f"Item: prompt_id={prompt_id}, prompt={prompt}, rubric={rubric}")

        prompt_outdir = IMAGES_DIR / prompt_id / trial_id
        prompt_outdir.mkdir(parents=True, exist_ok=True)

        for round in range(ROUNDS):
            print(f"\tRound {round}")
            round_outdir = prompt_outdir / f"round{round}"

            # Generate images using our method
            batch_images: list[Image.Image] = policy.generate(
                prompt,
                sampling_parameters={
                    "batch_size": BATCH_SIZE,
                    "num_inference_steps": 20,
                    "guidance_scale": 7.5,
                    "width": 512,
                    "height": 512,
                }
            )
            
            # Score and record metrics for each image in the batch
            user_scores = []
            for image_idx, image in enumerate(batch_images):
                image_path = round_outdir / f"image{image_idx}.png"
                
                user_score = score_image(image, rubric)
                user_scores.append(user_score)
                
                metrics.log(
                    prompt_id,
                    trial_id,
                    round,
                    image_idx,
                    image_path,
                    user_score,
                    image,
                    extra_info={
                        "prompt": prompt,
                    }
                )
                save_image(image, image_path)

            # Select favorite (index)
            chosen_image_idx = select(user_scores, strategy=SELECTION_STRATEGY)
            metrics.mark_chosen(
                prompt_id,
                trial_id,
                round,
                chosen_image_idx,
            )

            # Update policy
            policy.update(
                feedback={
                    # nothing for now
                }
            )

            print(f"\t\tScores: {user_scores}")
            print(f"\t\tChosen index (using {SELECTION_STRATEGY}): {chosen_image_idx}")
            
            
# Save metrics
metrics.save_csv(str(METRICS_CSV))
print("Saved metrics to", METRICS_CSV)